In [ ]:
## Magic to reload modules between cells
%load_ext autoreload
%autoreload 2

## GPU setup
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.device(device)
SEED=12345
_=torch.manual_seed(SEED)

In [ ]:
## Point to the right inputs

## Don't use the full dataset...
nevents = 100000

## Where to get the trained models from
state_file_dir = "$PSCRATCH" 
state_file_name = "state_GENIE10cNuMIME_ResNet50v1baselinev2NORM0max_PROJ00threebn_EXP4096_lars0.1WGT1E-6HEAD1_1024_50_1AUGv0l_2M_N2_NULARVICReg_LRSCALE0.03.pth"
state_file_path = f"{state_file_dir}/{state_file_name}"

## The dataset used for training
root_data_dir = "$PSCRATCH/NULARBOX"
nom_data_set = "GENIE10c_NuMIME_CCCONT256"
nom_data_dir = f"{root_data_dir}/{nom_data_set}"

In [ ]:
## Get the dataset
from larch.analysis.dataset_utils import get_dataset
from larch.datasets.nularbox.augmentations_2d import get_transform
from larch.analysis.model_utils import get_models_from_checkpoint

## Get the trained models
encoder, heads, args = get_models_from_checkpoint(state_file_path)

## The the augmentation set used in training
nom_transform = get_transform(256, "no_aug", 1.0)

## Load the datasets
nom_dataset, nom_loader = get_dataset(nom_data_dir, nevents, nom_transform)

In [ ]:
## Pass the images through the encoder
from larch.analysis.dataset_utils import image_loop

## Return hidden exposes the intermediate and final layers from the projection head, but is memory intensive
nom_processed = image_loop(encoder, heads, nom_loader, device, return_hidden=False, detailed_info=True)

In [ ]:
from larch.analysis.tsne_utils import compute_tsne_skl, compute_tsne_cuml
from larch.analysis.geometry_utils import preprocess_embeddings
import numpy as np

## PCA and basic transformation of the encoded vectors
## Note that the encoded vector magnitude matters so we're not using cosine similarity here
ntsne_max=50000
X_pca = preprocess_embeddings(
    nom_processed['encoder'],
    pca=50,
    drop_first_pca=False,
    center=True,
    normalize_before_pca=False,
    normalize_after_pca=False,
)

tsne_algo="cuml"

if tsne_algo == "skl":
    ## tSNE using the sklearn implementation 
    tsne_results = compute_tsne_skl(X_pca[:ntsne_max], 
                                    perp=150, exag=20,
                                    lr='auto', metric="euclidean")
else:
    ## tSNE using the cuML implementation: 
    ## barnes_hut is slower than fft, but fft has some weird behaviour
    ## like producing visual blocks through interpolation
    tsne_results = compute_tsne_cuml(X_pca, method='barnes_hut',
                                     perp=150, exag=20,
                                     metric="euclidean")

In [ ]:
from larch.analysis.tsne_utils import plot_tsne, plot_particle_tsne_block

## plot_tsne will make a single plot, these are convenience functions to dump a few which are of interest
print("Truth particle labels:")
plot_particle_tsne_block(tsne_results, nom_processed['labels']['particle_truth'])
print("Visible particle labels:")
plot_particle_tsne_block(tsne_results, nom_processed['labels']['particle_visible'])


In [ ]:
from larch.analysis.tsne_utils import plot_summary_tsne_block

## Another fun block
plot_summary_tsne_block(tsne_results, nom_processed)

In [ ]:
from larch.analysis.plotting_utils import plot_metric_by_label
from larch.datasets.nularbox.truth_labels import Mode, Topology, CCTopology
import numpy as np

## There are also a bunch of convenience plotting functions like:
plot_metric_by_label(np.log10(nom_processed['nhits']),
                     nom_processed['labels']['event']['cctopology_truth'],
                     label_enum=CCTopology,
                     xtitle=r"log$_{10}$(N. hits)", logy=False)